# PR11: SVM with Cross Validation and Grid Search

**Aim:** Train an SVM on the Pima Diabetes dataset (predict if a patient has diabetes), check it with cross validation, tune it with Grid Search and compare before and after.

**SVM** draws the boundary (hyperplane) that has the **maximum margin** between the two classes.

| Parameter | Meaning |
|---|---|
| `C` | Penalty for mistakes. Large C means a strict, tight fit (can overfit). |
| `kernel` | Shape of the boundary: `linear` or `rbf` (curved). |
| `gamma` | How far one point's influence reaches (for rbf). |

## Viva points

- **Cross validation** gives a more reliable score than a single train/test split.
- **Grid Search** = brute force over a grid of hyperparameters. `GridSearchCV` also does cross validation for each combination.
- **Hyperparameters** are set before training (C, kernel, gamma). **Parameters** are learned during training (weights).
- Kernel trick: lets SVM draw curved boundaries by mapping data to a higher dimension without computing it.
- SVM needs **scaled** features.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# 1. Load dataset
df = pd.read_csv("car_evaluation.csv", header=None)

df.columns = ["buying", "maint", "doors", "persons",
              "lug_boot", "safety", "class"]

# 2. Separate input and output
X = pd.get_dummies(df.drop("class", axis=1))
y = df["class"].astype("category").cat.codes

# 3. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Create models
models = {
    "SVM": make_pipeline(StandardScaler(), SVC()),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

# 5. Hyperparameters for Grid Search
params = {
    "SVM": {
        "svc__C": [1, 10],
        "svc__kernel": ["linear", "rbf"]
    },
    "Random Forest": {
        "n_estimators": [50, 100],
        "max_depth": [None, 10]
    },
    "AdaBoost": {
        "n_estimators": [50, 100],
        "learning_rate": [0.5, 1]
    }
}

# 6. Train, Cross Validate and Optimize
for name, model in models.items():

    # Before optimization
    model.fit(X_train, y_train)
    before = accuracy_score(y_test, model.predict(X_test))

    # Cross Validation
    cv = cross_val_score(model, X_train, y_train, cv=3).mean()

    # Grid Search
    grid = GridSearchCV(model, params[name], cv=3)
    grid.fit(X_train, y_train)

    # After optimization
    after = accuracy_score(y_test, grid.best_estimator_.predict(X_test))

    print("\n", name)
    print("Before Optimization:", round(before, 4))
    print("Cross Validation:", round(cv, 4))
    print("Best Parameters:", grid.best_params_)
    print("After Optimization:", round(after, 4))